Step 0: Loading the file from the same folder as where the code is

In [2]:
"""
Before running, run 'pip install pandas pyarrow numpy matplotlib pyfixest'

Step 1: Import the sector-level trade data and run basic checks.

Input : trade_sector.csv  (in the same folder as this script)
Output: trade.parquet      (same folder; fast to reload in later steps)
"""

from pathlib import Path

import pandas as pd

# Folder this script is in, so it works no matter where it is run from
try:
    HERE = Path(__file__).resolve().parent
except NameError:
    HERE = Path.cwd()
RAW_FILE = HERE / "trade_sector.csv"
OUT_FILE = HERE / "trade.parquet"

# --- 1. Read the CSV ------------------------------------------------------
# Country and sector columns repeat a lot, so "category" saves a lot of memory.
dtypes = {
    "exporter_iso3_dynamic": "category",
    "importer_iso3_dynamic": "category",
    "year": "int16",
    "broad_sector": "category",
    "trade": "float64",
    "n_ind": "int16",
    "n_u": "int16",
}
df = pd.read_csv(RAW_FILE, dtype=dtypes)

# Shorter column names for the rest of the project
df = df.rename(
    columns={
        "exporter_iso3_dynamic": "exporter",
        "importer_iso3_dynamic": "importer",
        "broad_sector": "sector",
    }
)

# --- 2. Flags we will need later ------------------------------------------
# n_ind = number of underlying industries in the row,
# n_u   = number of those industries with no reported data.
# If every industry is unreported, a 0 is a missing value, not a true zero.
df["domestic"] = (df["exporter"].astype(str) == df["importer"].astype(str))
df["all_missing"] = df["n_u"] == df["n_ind"]
df["true_zero"] = (df["trade"] == 0) & ~df["all_missing"]

# Unreported flows are unknown, not zero: set them to NaN but keep the rows.
# PPML drops NaN outcomes automatically; plots show them as gaps.
df["trade_raw"] = df["trade"]
df.loc[df["all_missing"], "trade"] = float("nan")

# --- 3. Basic checks --------------------------------------------------------
print(f"Rows: {len(df):,}")
print(f"Years: {df['year'].min()}-{df['year'].max()}")
print(f"Exporters: {df['exporter'].nunique()}, importers: {df['importer'].nunique()}")
print("\nRows per sector:")
print(df["sector"].value_counts())

dupes = df.duplicated(["exporter", "importer", "year", "sector"]).sum()
print(f"\nDuplicate exporter-importer-year-sector rows: {dupes}")
print("\nZeros in the raw data and how they were recoded:")
print(f"  zeros in raw file        : {(df['trade_raw'] == 0).mean():.1%} of rows")
print(f"  set to NaN (all missing) : {df['trade'].isna().mean():.1%}")
print(f"  true zeros kept          : {df['true_zero'].mean():.1%}")

# Croatia's domestic flows by sector (needed for structural gravity)
hrv_dom = (
    df[df["domestic"] & (df["exporter"] == "HRV")]
    .pivot_table(index="year", columns="sector", values="trade", observed=True)
    .round(0)
)
print("\nCroatia domestic trade (X_ii) by sector:")
print(hrv_dom)

# --- 4. Save ------------------------------------------------------------------
df.to_parquet(OUT_FILE, index=False)
print(f"\nSaved to {OUT_FILE}")


Rows: 1,991,999
Years: 2000-2019
Exporters: 250, importers: 249

Rows per sector:
sector
Manufacturing        895100
Agriculture          562268
Mining and Energy    432017
Services             102614
Name: count, dtype: int64

Duplicate exporter-importer-year-sector rows: 0

Zeros in the raw data and how they were recoded:
  zeros in raw file        : 41.5% of rows
  set to NaN (all missing) : 40.9%
  true zeros kept          : 0.6%

Croatia domestic trade (X_ii) by sector:
sector  Agriculture  Manufacturing  Mining and Energy  Services
year                                                           
2000          594.0            NaN                NaN  128643.0
2001          649.0            NaN                NaN  142383.0
2002          723.0            NaN                NaN  171026.0
2003          732.0            NaN                NaN  235575.0
2004          979.0            NaN                NaN  274529.0
2005          988.0            NaN                NaN  320061.0
2006    

Step 2: keep Manufacturing only because of missing data (only fully complete category), drop NAs and keep true zeros, uniting ROM and ROU, dropping Serbia and Montenegro (as a united country), unifying SRB and SRB.X

In [3]:
"""
Step 2: Build the estimation sample.

Keeps international manufacturing flows, 2000-2019, harmonises the Romania
code, drops unreported flows and saves the sample used in all later steps.

Input : trade.parquet         (from 01_import.py, same folder as this script)
Output: sample_manuf.parquet  (same folder)
"""

from pathlib import Path

import pandas as pd

# Folder this script is in (falls back to the current folder in Jupyter)
try:
    HERE = Path(__file__).resolve().parent
except NameError:
    HERE = Path.cwd()

IN_FILE = HERE / "trade.parquet"
OUT_FILE = HERE / "sample_manuf.parquet"

# --- 1. Load and keep international manufacturing flows --------------------
df = pd.read_parquet(IN_FILE)

# Plain strings are easier to work with than categories from here on
for col in ["exporter", "importer", "sector"]:
    df[col] = df[col].astype(str)

df = df[(df["sector"] == "Manufacturing") & ~df["domestic"]].copy()
print(f"International manufacturing rows: {len(df):,}")

# --- 2. Harmonise country codes ----------------------------------------------
# Romania: ISO code changed from ROM to ROU in 2002. Same country, new label.
# Serbia:  SRB (2006-2007) and SRB.X (2008-2019, presumably without Kosovo)
#          are merged into one Serbia series, SRB.
# SCG:     Serbia and Montenegro as one country (2003-2005). Dropped, so that
#          Serbia and Montenegro are each observed only as separate countries,
#          from 2006 on.
# Other codes that reflect border changes elsewhere are kept as they are.
RECODE = {"ROM": "ROU", "SRB.X": "SRB"}
DROP = ["SCG"]

df["exporter"] = df["exporter"].replace(RECODE)
df["importer"] = df["importer"].replace(RECODE)

n_scg = (df["exporter"].isin(DROP) | df["importer"].isin(DROP)).sum()
df = df[~(df["exporter"].isin(DROP) | df["importer"].isin(DROP))].copy()
print(f"Dropped {n_scg:,} rows involving {DROP}")

dupes = df.duplicated(["exporter", "importer", "year"]).sum()
assert dupes == 0, f"{dupes} duplicate pair-years after recoding"

# --- 3. Drop unreported flows, keep true zeros ---------------------------------
n_before = len(df)
df = df[df["trade"].notna()].copy()
print(f"Dropped {n_before - len(df):,} unreported (NaN) rows")
print(f"Kept {len(df):,} rows, of which {(df['trade'] == 0).sum()} true zeros")

# --- 4. Tidy up --------------------------------------------------------------
df["pair"] = df["exporter"] + "_" + df["importer"]
df = df[["exporter", "importer", "pair", "year", "trade", "n_ind", "n_u"]]
df = df.sort_values(["exporter", "importer", "year"]).reset_index(drop=True)

# --- 5. Checks -----------------------------------------------------------------
years_per_pair = df.groupby("pair")["year"].nunique()
value_per_pair = df.groupby("pair")["trade"].sum()
complete = years_per_pair == df["year"].nunique()

print(f"\nYears: {df['year'].min()}-{df['year'].max()}")
print(f"Exporters: {df['exporter'].nunique()}, importers: {df['importer'].nunique()}")
print(f"Pairs: {len(years_per_pair):,}")
print(f"  observed in all years: {complete.sum():,} ({complete.mean():.1%})")
print(f"  their share of trade value: {value_per_pair[complete].sum() / value_per_pair.sum():.1%}")
print(f"  observed in one year only: {(years_per_pair == 1).sum():,} "
      "(dropped by PPML later: a pair fixed effect fits them perfectly)")

# Croatia's partners: years observed per direction
EU = ["AUT", "BEL", "BGR", "CYP", "CZE", "DEU", "DNK", "ESP", "EST", "FIN",
      "FRA", "GBR", "GRC", "HUN", "IRL", "ITA", "LTU", "LUX", "LVA", "MLT",
      "NLD", "POL", "PRT", "ROU", "SVK", "SVN", "SWE"]
CEFTA = ["ALB", "BIH", "MDA", "MKD", "MNE", "SRB"]

hrv = df[(df["exporter"] == "HRV") | (df["importer"] == "HRV")].copy()
hrv["partner"] = hrv["exporter"].where(hrv["exporter"] != "HRV", hrv["importer"])
hrv["direction"] = (hrv["exporter"] == "HRV").map({True: "HRV exports", False: "HRV imports"})

coverage = hrv.pivot_table(index="partner", columns="direction", values="year",
                           aggfunc="nunique")
print("\nCroatia-EU pairs observed in all 20 years (both directions): "
      f"{(coverage.loc[coverage.index.isin(EU)].min(axis=1) == 20).sum()} of {len(EU)}")
print("\nCroatia-CEFTA coverage (years observed):")
print(coverage.loc[coverage.index.isin(CEFTA)])

# Serbia: check that the merged series has no break at the 2007/2008 code change
srb = df[df["exporter"] == "SRB"].groupby("year")["trade"].sum()
hrv_srb = df[(df["exporter"] == "HRV") & (df["importer"] == "SRB")].set_index("year")["trade"]
print("\nSerbia around the code change (million USD):")
print(pd.DataFrame({"Serbia total exports": srb, "Croatia exports to Serbia": hrv_srb})
      .loc[2006:2010].round(0))

# --- 6. Save -------------------------------------------------------------------
df.to_parquet(OUT_FILE, index=False)
print(f"\nSaved to {OUT_FILE}")


International manufacturing rows: 893,226
Dropped 1,172 rows involving ['SCG']
Dropped 294,253 unreported (NaN) rows
Kept 597,801 rows, of which 46 true zeros

Years: 2000-2019
Exporters: 239, importers: 238
Pairs: 45,106
  observed in all years: 17,465 (38.7%)
  their share of trade value: 99.0%
  observed in one year only: 3,100 (dropped by PPML later: a pair fixed effect fits them perfectly)

Croatia-EU pairs observed in all 20 years (both directions): 27 of 27

Croatia-CEFTA coverage (years observed):
direction  HRV exports  HRV imports
partner                            
ALB               20.0         20.0
BIH               20.0         20.0
MDA               20.0         20.0
MKD               20.0         20.0
MNE               14.0         14.0
SRB               14.0         14.0

Serbia around the code change (million USD):
      Serbia total exports  Croatia exports to Serbia
year                                                 
2006                5945.0                     

Step 3: Dummies

In [1]:
"""
Step 3: Add the treatment dummies (specification agreed with partner).

  HRV_EU    = 1  Croatia-EU pair, from Croatia's accession (2013 on)
  HRV_SAA   = 1  Croatia-EU pair, from 2002 (SAA Interim Agreement) and it
                 STAYS 1 after accession (cumulative coding). So the HRV_EU
                 coefficient is the effect of membership on top of the SAA.
                 2004/2007 members count from their own EU accession, since
                 the SAA only covered them once they were in the EU.
  EU        = 1  both countries EU members, pairs without Croatia
                 (captures the 2004 and 2007 enlargements)
  HRV_CEFTA = 1  Croatia and partner both in CEFTA (2003-2012).
                 Not in the main specification; kept for robustness.

2013 is half treated (accession 1 July 2013). It is coded here as the first
treated year and dropped in the main regression in 04_estimate.py, so it can
still be reinserted for the robustness checks.

Part-year rule: a year counts if the agreement applies for at least half of it.

Input : sample_manuf.parquet   (from 02_build_sample.py, same folder)
Output: sample_dummies.parquet (same folder)
"""

from pathlib import Path

import numpy as np
import pandas as pd

# Folder this script is in (falls back to the current folder in Jupyter)
try:
    HERE = Path(__file__).resolve().parent
except NameError:
    HERE = Path.cwd()

IN_FILE = HERE / "sample_manuf.parquet"
OUT_FILE = HERE / "sample_dummies.parquet"

# --- 1. Membership dates ------------------------------------------------------
# First year a country counts as an EU member. EU15 joined before the sample;
# the UK left in 2020, after the sample, so it is a member throughout.
EU15 = ["AUT", "BEL", "DEU", "DNK", "ESP", "FIN", "FRA", "GBR", "GRC", "IRL",
        "ITA", "LUX", "NLD", "PRT", "SWE"]
EU2004 = ["CYP", "CZE", "EST", "HUN", "LTU", "LVA", "MLT", "POL", "SVK", "SVN"]
EU2007 = ["BGR", "ROU"]

EU_YEAR = {c: 1995 for c in EU15}          # any year before 2000 works
EU_YEAR.update({c: 2004 for c in EU2004})
EU_YEAR.update({c: 2007 for c in EU2007})
EU_YEAR["HRV"] = 2013

SAA_START = 2002           # Interim Agreement applied from 2002

# CEFTA membership of Croatia's partners: (first year, last year)
CEFTA_YEARS = {
    "POL": (1993, 2003), "HUN": (1993, 2003), "CZE": (1993, 2003),
    "SVK": (1993, 2003), "SVN": (1996, 2003),
    "ROU": (1997, 2006), "BGR": (1999, 2006),
    "MKD": (2006, 9999),
    "ALB": (2007, 9999), "BIH": (2007, 9999), "MDA": (2007, 9999),
    "MNE": (2007, 9999), "SRB": (2007, 9999),
}
HRV_CEFTA_YEARS = (2003, 2012)   # Croatia: joined March 2003, left 30 June 2013

# --- 2. Load -------------------------------------------------------------------
df = pd.read_parquet(IN_FILE)
year = df["year"].to_numpy()


def is_eu(codes: pd.Series) -> np.ndarray:
    """True where the country is an EU member in that row's year."""
    first = codes.map(EU_YEAR).fillna(9999).to_numpy()
    return first <= year


def in_cefta(codes: pd.Series) -> np.ndarray:
    """True where the partner is a CEFTA member in that row's year."""
    start = codes.map({c: s for c, (s, e) in CEFTA_YEARS.items()}).fillna(9999).to_numpy()
    end = codes.map({c: e for c, (s, e) in CEFTA_YEARS.items()}).fillna(0).to_numpy()
    return (start <= year) & (year <= end)


# --- 3. Croatian pairs -------------------------------------------------------------
hrv_pair = ((df["exporter"] == "HRV") | (df["importer"] == "HRV")).to_numpy()
partner = df["importer"].where(df["exporter"] == "HRV", df["exporter"])
partner_eu = is_eu(partner)

df["HRV_EU"] = (hrv_pair & partner_eu & (year >= EU_YEAR["HRV"])).astype(int)
df["HRV_SAA"] = (hrv_pair & partner_eu & (year >= SAA_START)).astype(int)
df["HRV_CEFTA"] = (hrv_pair & in_cefta(partner)
                   & (year >= HRV_CEFTA_YEARS[0])
                   & (year <= HRV_CEFTA_YEARS[1])).astype(int)

# --- 4. Pairs without Croatia: both in the EU --------------------------------------
df["EU"] = (~hrv_pair & is_eu(df["exporter"]) & is_eu(df["importer"])).astype(int)

# --- 5. Checks -------------------------------------------------------------------------
# HRV_EU must imply HRV_SAA (cumulative coding), and CEFTA never overlaps them
assert (df["HRV_EU"] <= df["HRV_SAA"]).all(), "HRV_EU = 1 without HRV_SAA = 1"
assert not ((df["HRV_CEFTA"] == 1) & (df["HRV_SAA"] == 1)).any(), "CEFTA overlaps SAA/EU"

print("Rows with each dummy = 1:")
print(df[["HRV_EU", "HRV_SAA", "EU", "HRV_CEFTA"]].sum().to_string())

hrv = df[hrv_pair].assign(partner=partner[hrv_pair])


def span(years: pd.Series) -> str:
    return f"{years.min()}-{years.max()}" if len(years) else "-"


dummy_cols = ["HRV_SAA", "HRV_EU", "HRV_CEFTA"]
summary = pd.DataFrame({
    d: hrv[hrv[d] == 1].groupby("partner")["year"].apply(span) for d in dummy_cols
})
summary = summary[summary.notna().any(axis=1)].fillna("-")
summary["observed"] = hrv.groupby("partner")["year"].apply(span)
order = EU15 + EU2004 + EU2007 + ["MKD", "ALB", "BIH", "MDA", "MNE", "SRB"]
print("\nYears each dummy = 1, by Croatian partner:")
print(summary.reindex([c for c in order if c in summary.index]).to_string())

# --- 6. Save ---------------------------------------------------------------------------
df.to_parquet(OUT_FILE, index=False)
print(f"\nSaved to {OUT_FILE}")


Rows with each dummy = 1:
HRV_EU         378
HRV_SAA        912
EU           11766
HRV_CEFTA      100

Years each dummy = 1, by Croatian partner:
           HRV_SAA     HRV_EU  HRV_CEFTA   observed
partner                                            
AUT      2002-2019  2013-2019          -  2000-2019
BEL      2002-2019  2013-2019          -  2000-2019
DEU      2002-2019  2013-2019          -  2000-2019
DNK      2002-2019  2013-2019          -  2000-2019
ESP      2002-2019  2013-2019          -  2000-2019
FIN      2002-2019  2013-2019          -  2000-2019
FRA      2002-2019  2013-2019          -  2000-2019
GBR      2002-2019  2013-2019          -  2000-2019
GRC      2002-2019  2013-2019          -  2000-2019
IRL      2002-2019  2013-2019          -  2000-2019
ITA      2002-2019  2013-2019          -  2000-2019
LUX      2002-2019  2013-2019          -  2000-2019
NLD      2002-2019  2013-2019          -  2000-2019
PRT      2002-2019  2013-2019          -  2000-2019
SWE      2002-2019  20

4. Regression

In [2]:
"""
Step 4: Estimate the gravity equation with PPML (main specification).

X_ijt = exp( pi_it + chi_jt + mu_ij
             + b1 HRV_EU + b2 HRV_SAA + b3 EU ) * e_ijt

pi_it  exporter-year fixed effects
chi_jt importer-year fixed effects
mu_ij  pair fixed effects
Standard errors are clustered by pair.

HRV_SAA is coded cumulatively, so b1 is the effect of EU membership on top
of the SAA. 2013 (half treated) is dropped for all pairs.

Input : sample_dummies.parquet (from 03_dummies.py, same folder)
Output: results_ppml.csv       (same folder)
"""

from pathlib import Path

import numpy as np
import pandas as pd
import pyfixest as pf
from scipy import stats

# Folder this script is in (falls back to the current folder in Jupyter)
try:
    HERE = Path(__file__).resolve().parent
except NameError:
    HERE = Path.cwd()

IN_FILE = HERE / "sample_dummies.parquet"
OUT_FILE = HERE / "results_ppml.csv"

# --- 1. Load, drop 2013, build the fixed-effect identifiers ------------------
df = pd.read_parquet(IN_FILE)
df = df[df["year"] != 2013].copy()

df["exp_year"] = df["exporter"] + "_" + df["year"].astype(str)
df["imp_year"] = df["importer"] + "_" + df["year"].astype(str)

print(f"Observations (2013 dropped): {len(df):,}")
print(f"Fixed effects: {df['exp_year'].nunique():,} exporter-year, "
      f"{df['imp_year'].nunique():,} importer-year, {df['pair'].nunique():,} pair")

# --- 2. PPML ----------------------------------------------------------------
TREATMENTS = ["HRV_EU", "HRV_SAA", "EU"]
formula = "trade ~ " + " + ".join(TREATMENTS) + " | exp_year + imp_year + pair"

fit = pf.fepois(formula, data=df, vcov={"CRV1": "pair"})
fit.summary()

# --- 3. Results table -----------------------------------------------------------
coef = fit.coef()
se = fit.se()
res = pd.DataFrame({
    "coef": coef,
    "se": se,
    "p_value": 2 * stats.norm.sf(np.abs(coef / se)),
    "pct_effect": (np.exp(coef) - 1) * 100,   # (e^b - 1) x 100
})

print("\nResults (pct_effect = (exp(coef) - 1) x 100):")
print(res.round(4).to_string())
print(f"\nObservations used: {fit._N:,}  "
      "(PPML drops fixed effects whose trade is perfectly predicted, e.g. singletons)")

# --- 4. Save -------------------------------------------------------------------
res.to_csv(OUT_FILE)
print(f"\nSaved to {OUT_FILE}")


Observations (2013 dropped): 566,274
Fixed effects: 4,418 exporter-year, 4,417 importer-year, 44,934 pair


c:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:346: UserWarning: 3166 singleton fixed effect(s) dropped from the model.
  warnings.warn(f"{n_dropped} {reason} dropped from the model.")


###

Estimation:  Poisson
Dep. var.: trade, Fixed effects: exp_year+imp_year+pair
sample: None = all
Inference:  CRV1
Observations:  563108

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| HRV_EU        |      0.501 |        0.105 |     4.772 |      0.000 |  0.295 |   0.707 |
| HRV_SAA       |     -0.101 |        0.076 |    -1.331 |      0.183 | -0.249 |   0.048 |
| EU            |      0.010 |        0.041 |     0.236 |      0.814 | -0.070 |   0.089 |
---
Deviance: 10296948.374 

Results (pct_effect = (exp(coef) - 1) x 100):
               coef      se  p_value  pct_effect
Coefficient                                     
HRV_EU       0.5012  0.1050   0.0000     65.0691
HRV_SAA     -0.1006  0.0756   0.1832     -9.5730
EU           0.0096  0.0406   0.8137      0.9612

Observations used: 563,108  (PPML drops fixed effects whose trade is perfectly predicted